# 🔬 RobustLens: Independent ImageNet-V2 External Replication Study
## 10-Synset External Replication on MatchedFrequency, Threshold0.7, and TopImages

- **Objective:** Evaluate external generalization of distortion fingerprints across independently collected natural photographs without parameter retuning.
- **Variants:** MatchedFrequency ($N=100$), Threshold0.7 ($N=100$), TopImages ($N=100$). Pooled: $N=300$ images.
- **Total Passes:** $300 \times 19 \times 4 = 22{,}800$ forward passes ($1{,}200$ clean + $21{,}600$ corrupted).
- **Protocol:** Strictly frozen parameters from calibrated 3-level common support.


In [ ]:
# 1. Environment, Dependencies & GPU Verification
import os, sys, time, json, math, io, tarfile, zipfile, urllib.request, hashlib, subprocess
from pathlib import Path
import numpy as np
import pandas as pd
import scipy.signal
from PIL import Image
import torch
import torch.nn.functional as F
import torchvision
import torchvision.transforms as T
import torchvision.transforms.functional as TF

# Ensure lpips and timm are installed
for pkg in ['lpips', 'timm']:
    try:
        __import__(pkg)
    except ImportError:
        print(f'Installing {pkg}...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

import lpips
import timm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Active Device: {device}')
if torch.cuda.is_available():
    print(f'GPU Device Name: {torch.cuda.get_device_name(0)}')
    print(f'GPU Total Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB')


In [ ]:
# 2. ImageNet-V2 Dataset Loading for 10 Target Synsets
TARGET_CLASSES = {
    0: ('n01440764', 'tench'),
    217: ('n02102040', 'English springer'),
    482: ('n02979186', 'cassette player'),
    491: ('n03000684', 'chain saw'),
    497: ('n03028079', 'church'),
    566: ('n03394916', 'French horn'),
    569: ('n03417042', 'garbage truck'),
    571: ('n03425413', 'gas pump'),
    574: ('n03445777', 'golf ball'),
    701: ('n03888257', 'parachute'),
}

VARIANTS = ['MatchedFrequency', 'Threshold0.7', 'TopImages']
v2_dataset = []
data_root = Path('imagenet_v2_extracted')
data_root.mkdir(exist_ok=True)

kaggle_input_dir = Path('/kaggle/input/imagenet-v2-10synsets')
dataset_loaded = False

if kaggle_input_dir.exists():
    print(f'Found mounted Kaggle dataset at {kaggle_input_dir}!')
    for var_name in VARIANTS:
        zip_p = kaggle_input_dir / f'{var_name}.zip'
        target_v_dir = data_root / var_name
        target_v_dir.mkdir(exist_ok=True)
        if zip_p.exists():
            print(f'Extracting {zip_p.name}...')
            with zipfile.ZipFile(zip_p, 'r') as zf:
                zf.extractall(target_v_dir)
        for c_idx, (wnid, cname) in TARGET_CLASSES.items():
            img_paths = list(target_v_dir.rglob(f'{c_idx}/*.jpeg'))
            for img_p in img_paths:
                v2_dataset.append({
                    'variant': var_name,
                    'class_idx': c_idx,
                    'wordnet_id': wnid,
                    'class_name': cname,
                    'filepath': str(img_p),
                    'filename': img_p.name,
                })
    if len(v2_dataset) == 300:
        print(f'Successfully loaded all {len(v2_dataset)} images from mounted Kaggle dataset!')
        dataset_loaded = True
    else:
        print(f'Mounted dataset found {len(v2_dataset)} images; falling back to streaming...')
        v2_dataset = []

if not dataset_loaded:
    print('Streaming ImageNet-V2 variants from Hugging Face...')
    HF_URLS = {
        'MatchedFrequency': 'https://huggingface.co/datasets/vaishaal/ImageNetV2/resolve/main/imagenetv2-matched-frequency.tar.gz',
        'Threshold0.7': 'https://huggingface.co/datasets/vaishaal/ImageNetV2/resolve/main/imagenetv2-threshold0.7.tar.gz',
        'TopImages': 'https://huggingface.co/datasets/vaishaal/ImageNetV2/resolve/main/imagenetv2-top-images.tar.gz',
    }
    for var_name, url in HF_URLS.items():
        var_dir = data_root / var_name
        var_dir.mkdir(parents=True, exist_ok=True)
        print(f'Streaming {var_name}...')
        req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
        res = urllib.request.urlopen(req)
        try:
            t = tarfile.open(mode='r|gz', fileobj=res)
        except Exception:
            res.close()
            res = urllib.request.urlopen(req)
            t = tarfile.open(mode='r|', fileobj=res)
        var_count = 0
        class_counts = {c: 0 for c in TARGET_CLASSES}
        for m in t:
            parts = m.name.split('/')
            if len(parts) >= 3 and parts[-2].isdigit() and parts[-1].endswith('.jpeg'):
                c_idx = int(parts[-2])
                if c_idx in TARGET_CLASSES:
                    c_dir = var_dir / str(c_idx)
                    c_dir.mkdir(exist_ok=True)
                    out_p = c_dir / parts[-1]
                    f_obj = t.extractfile(m)
                    if f_obj is not None:
                        content = f_obj.read()
                        out_p.write_bytes(content)
                        wnid, cname = TARGET_CLASSES[c_idx]
                        v2_dataset.append({
                            'variant': var_name,
                            'class_idx': c_idx,
                            'wordnet_id': wnid,
                            'class_name': cname,
                            'filepath': str(out_p),
                            'filename': parts[-1],
                        })
                        class_counts[c_idx] += 1
                        var_count += 1
                        if var_count >= 100 and all(v == 10 for v in class_counts.values()):
                            break
        res.close()
        print(f'  Extracted {var_count} images for {var_name}.')

# Sort deterministically
v2_dataset = sorted(v2_dataset, key=lambda x: (x['variant'], x['class_idx'], x['filename']))
for idx, item in enumerate(v2_dataset):
    item['image_id'] = idx
print(f'Total Validated ImageNet-V2 Dataset: {len(v2_dataset)} unique images across 3 variants.')


In [ ]:
# 3. Metrics, Descriptors & Frozen Distortion Transforms
def compute_psnr(clean: torch.Tensor, corrupt: torch.Tensor) -> float:
    mse = torch.mean((clean.float() - corrupt.float()) ** 2).item()
    if mse < 1e-10:
        return 100.0
    return float(10.0 * math.log10(1.0 / mse))

def _create_ssim_kernel(k: int = 11, sigma: float = 1.5) -> torch.Tensor:
    coords = torch.arange(k, dtype=torch.float32) - (k - 1) / 2.0
    g = torch.exp(-(coords ** 2) / (2.0 * sigma ** 2))
    g = g / g.sum()
    k2d = torch.outer(g, g)
    return k2d.expand(3, 1, k, k)

_SSIM_K = _create_ssim_kernel().to(device)

def compute_ssim_and_local_map(clean: torch.Tensor, corrupt: torch.Tensor):
    global _SSIM_K
    if clean.ndim == 3:
        clean = clean.unsqueeze(0)
    if corrupt.ndim == 3:
        corrupt = corrupt.unsqueeze(0)
    clean = clean.float().to(device)
    corrupt = corrupt.float().to(device)
    c1 = (0.01) ** 2
    c2 = (0.03) ** 2
    mu1 = F.conv2d(clean, _SSIM_K, padding=5, groups=3)
    mu2 = F.conv2d(corrupt, _SSIM_K, padding=5, groups=3)
    mu1_sq, mu2_sq, mu1_mu2 = mu1.pow(2), mu2.pow(2), mu1 * mu2
    sigma1_sq = F.conv2d(clean * clean, _SSIM_K, padding=5, groups=3) - mu1_sq
    sigma2_sq = F.conv2d(corrupt * corrupt, _SSIM_K, padding=5, groups=3) - mu2_sq
    sigma12 = F.conv2d(clean * corrupt, _SSIM_K, padding=5, groups=3) - mu1_mu2
    ssim_map = ((2 * mu1_mu2 + c1) * (2 * sigma12 + c2)) / ((mu1_sq + mu2_sq + c1) * (sigma1_sq + sigma2_sq + c2) + 1e-8)
    return float(ssim_map.mean().item()), float(ssim_map.std().item())

def compute_distortion_fingerprints(clean_t: torch.Tensor, corrupt_t: torch.Tensor):
    clean_np = clean_t.squeeze(0).permute(1, 2, 0).cpu().numpy()
    corrupt_np = corrupt_t.squeeze(0).permute(1, 2, 0).cpu().numpy()
    gray_clean = clean_np.mean(axis=2).astype(np.float32)
    gray_corrupt = corrupt_np.mean(axis=2).astype(np.float32)
    delta = gray_corrupt - gray_clean
    H, W = delta.shape
    fft2 = np.fft.fft2(delta)
    power = np.abs(np.fft.fftshift(fft2)) ** 2
    y, x = np.ogrid[-H/2:H/2, -W/2:W/2]
    r_norm = np.sqrt(x**2 + y**2) / np.sqrt((H/2)**2 + (W/2)**2)
    total_power = power.sum() + 1e-12
    e_low = float(power[r_norm < 0.15].sum() / total_power)
    e_med = float(power[(r_norm >= 0.15) & (r_norm < 0.45)].sum() / total_power)
    e_high = float(power[r_norm >= 0.45].sum() / total_power)
    diff_h = np.abs(gray_corrupt[:, 1:] - gray_corrupt[:, :-1])
    mask_h = np.zeros(diff_h.shape[1], dtype=bool)
    mask_h[7::8] = True
    b_h = diff_h[:, mask_h].mean() if mask_h.sum() > 0 else 0.0
    i_h = diff_h[:, ~mask_h].mean() if (~mask_h).sum() > 0 else 1.0
    diff_v = np.abs(gray_corrupt[1:, :] - gray_corrupt[:-1, :])
    mask_v = np.zeros(diff_v.shape[0], dtype=bool)
    mask_v[7::8] = True
    b_v = diff_v[mask_v, :].mean() if mask_v.sum() > 0 else 0.0
    i_v = diff_v[~mask_v, :].mean() if (~mask_v).sum() > 0 else 1.0
    block_ratio = float(((b_h / (i_h + 1e-6)) + (b_v / (i_v + 1e-6))) / 2.0)
    sobel_x = np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=np.float32)
    sobel_y = np.array([[-1, -2, -1], [0, 0, 0], [1, 2, 1]], dtype=np.float32)
    edge_clean = np.sqrt(scipy.signal.convolve2d(gray_clean, sobel_x, mode='same')**2 + scipy.signal.convolve2d(gray_clean, sobel_y, mode='same')**2).mean()
    edge_corr = np.sqrt(scipy.signal.convolve2d(gray_corrupt, sobel_x, mode='same')**2 + scipy.signal.convolve2d(gray_corrupt, sobel_y, mode='same')**2).mean()
    delta_edge = float((edge_corr - edge_clean) / (edge_clean + 1e-6))
    _, std_local_ssim = compute_ssim_and_local_map(clean_t, corrupt_t)
    return {
        'fft_energy_low': round(e_low, 4),
        'fft_energy_med': round(e_med, 4),
        'fft_energy_high': round(e_high, 4),
        'jpeg_block_artifact': round(block_ratio, 4),
        'delta_edge_energy': round(delta_edge, 4),
        'std_local_ssim': round(std_local_ssim, 4),
    }

def apply_compound_sequence(img: torch.Tensor, order: str, noise_std: float, blur_sigma: float, jpeg_quality: int, noise_seed: int = 42):
    gen = torch.Generator().manual_seed(noise_seed)
    noise = torch.randn(img.shape, generator=gen, dtype=img.dtype) * max(0.0001, noise_std)
    cur = img
    k = max(3, int(2 * round(blur_sigma * 1.5) + 1))
    if k % 2 == 0: k += 1
    for char in order:
        if char == 'N':
            cur = torch.clamp(cur + noise, 0.0, 1.0)
        elif char == 'B':
            if blur_sigma > 0.05:
                cur = TF.gaussian_blur(cur, [k, k], [blur_sigma, blur_sigma])
        elif char == 'J':
            q = max(5, min(95, int(round(jpeg_quality))))
            pil_img = TF.to_pil_image(cur.cpu())
            buf = io.BytesIO()
            pil_img.save(buf, format='JPEG', quality=q)
            buf.seek(0)
            cur = TF.to_tensor(Image.open(buf))
    return cur


In [ ]:
# 4. Confirmatory Evaluation Loop on ImageNet-V2 (300 Images x 18 Conditions x 4 Models)
CALIBRATED_PARAMS = json.loads(r'''{
  "protocol": "RobustLens 6-Way Simultaneous Common-Support Calibration",
  "delineation_method": "Tolerance-Ball Union Density Estimator (Alpha=1.0)",
  "prespecified_tolerances": {
    "psnr_tolerance_db": 0.75,
    "ssim_tolerance": 0.035
  },
  "levels": {
    "Level_1_Mild": {
      "target_psnr": 27.2,
      "target_ssim": 0.81,
      "all_6_pass": true,
      "max_normalized_error": 0.327,
      "sequences": {
        "NBJ": {
          "final_op": "JPEG",
          "noise_std": 0.07,
          "blur_sigma": 1.0,
          "jpeg_quality": 75,
          "achieved_psnr": 27.22,
          "achieved_ssim": 0.82,
          "psnr_err": 0.02,
          "ssim_err": 0.0096,
          "within_tol": true
        },
        "NJB": {
          "final_op": "Blur",
          "noise_std": 0.07,
          "blur_sigma": 1.0,
          "jpeg_quality": 35,
          "achieved_psnr": 27.08,
          "achieved_ssim": 0.82,
          "psnr_err": -0.12,
          "ssim_err": 0.0099,
          "within_tol": true
        },
        "BNJ": {
          "final_op": "JPEG",
          "noise_std": 0.035,
          "blur_sigma": 1.0,
          "jpeg_quality": 55,
          "achieved_psnr": 27.1,
          "achieved_ssim": 0.803,
          "psnr_err": -0.1,
          "ssim_err": -0.0074,
          "within_tol": true
        },
        "BJN": {
          "final_op": "Noise",
          "noise_std": 0.015,
          "blur_sigma": 1.0,
          "jpeg_quality": 75,
          "achieved_psnr": 27.33,
          "achieved_ssim": 0.801,
          "psnr_err": 0.13,
          "ssim_err": -0.0089,
          "within_tol": true
        },
        "JNB": {
          "final_op": "Blur",
          "noise_std": 0.07,
          "blur_sigma": 1.0,
          "jpeg_quality": 90,
          "achieved_psnr": 27.26,
          "achieved_ssim": 0.807,
          "psnr_err": 0.06,
          "ssim_err": -0.0034,
          "within_tol": true
        },
        "JBN": {
          "final_op": "Noise",
          "noise_std": 0.015,
          "blur_sigma": 1.0,
          "jpeg_quality": 55,
          "achieved_psnr": 27.42,
          "achieved_ssim": 0.808,
          "psnr_err": 0.22,
          "ssim_err": -0.0023,
          "within_tol": true
        }
      }
    },
    "Level_2_Moderate": {
      "target_psnr": 23.8,
      "target_ssim": 0.65,
      "all_6_pass": true,
      "max_normalized_error": 0.333,
      "sequences": {
        "NBJ": {
          "final_op": "JPEG",
          "noise_std": 0.2,
          "blur_sigma": 1.0,
          "jpeg_quality": 75,
          "achieved_psnr": 23.69,
          "achieved_ssim": 0.647,
          "psnr_err": -0.11,
          "ssim_err": -0.0034,
          "within_tol": true
        },
        "NJB": {
          "final_op": "Blur",
          "noise_std": 0.2,
          "blur_sigma": 1.0,
          "jpeg_quality": 35,
          "achieved_psnr": 23.55,
          "achieved_ssim": 0.651,
          "psnr_err": -0.25,
          "ssim_err": 0.0013,
          "within_tol": true
        },
        "BNJ": {
          "final_op": "JPEG",
          "noise_std": 0.12,
          "blur_sigma": 0.5,
          "jpeg_quality": 10,
          "achieved_psnr": 23.75,
          "achieved_ssim": 0.646,
          "psnr_err": -0.05,
          "ssim_err": -0.0036,
          "within_tol": true
        },
        "BJN": {
          "final_op": "Noise",
          "noise_std": 0.015,
          "blur_sigma": 1.8,
          "jpeg_quality": 20,
          "achieved_psnr": 23.81,
          "achieved_ssim": 0.647,
          "psnr_err": 0.01,
          "ssim_err": -0.0031,
          "within_tol": true
        },
        "JNB": {
          "final_op": "Blur",
          "noise_std": 0.12,
          "blur_sigma": 1.0,
          "jpeg_quality": 10,
          "achieved_psnr": 23.96,
          "achieved_ssim": 0.646,
          "psnr_err": 0.16,
          "ssim_err": -0.0039,
          "within_tol": true
        },
        "JBN": {
          "final_op": "Noise",
          "noise_std": 0.015,
          "blur_sigma": 1.8,
          "jpeg_quality": 10,
          "achieved_psnr": 23.65,
          "achieved_ssim": 0.656,
          "psnr_err": -0.15,
          "ssim_err": 0.006,
          "within_tol": true
        }
      }
    },
    "Level_3_SevereCommon": {
      "target_psnr": 21.8,
      "target_ssim": 0.5,
      "all_6_pass": true,
      "max_normalized_error": 0.533,
      "sequences": {
        "NBJ": {
          "final_op": "JPEG",
          "noise_std": 0.2,
          "blur_sigma": 0.5,
          "jpeg_quality": 35,
          "achieved_psnr": 21.71,
          "achieved_ssim": 0.515,
          "psnr_err": -0.09,
          "ssim_err": 0.0155,
          "within_tol": true
        },
        "NJB": {
          "final_op": "Blur",
          "noise_std": 0.2,
          "blur_sigma": 0.5,
          "jpeg_quality": 20,
          "achieved_psnr": 21.65,
          "achieved_ssim": 0.515,
          "psnr_err": -0.15,
          "ssim_err": 0.0148,
          "within_tol": true
        },
        "BNJ": {
          "final_op": "JPEG",
          "noise_std": 0.07,
          "blur_sigma": 2.8,
          "jpeg_quality": 35,
          "achieved_psnr": 22.01,
          "achieved_ssim": 0.496,
          "psnr_err": 0.21,
          "ssim_err": -0.0042,
          "within_tol": true
        },
        "BJN": {
          "final_op": "Noise",
          "noise_std": 0.07,
          "blur_sigma": 0.5,
          "jpeg_quality": 20,
          "achieved_psnr": 22.03,
          "achieved_ssim": 0.486,
          "psnr_err": 0.23,
          "ssim_err": -0.0135,
          "within_tol": true
        },
        "JNB": {
          "final_op": "Blur",
          "noise_std": 0.12,
          "blur_sigma": 0.5,
          "jpeg_quality": 35,
          "achieved_psnr": 21.96,
          "achieved_ssim": 0.496,
          "psnr_err": 0.16,
          "ssim_err": -0.0043,
          "within_tol": true
        },
        "JBN": {
          "final_op": "Noise",
          "noise_std": 0.07,
          "blur_sigma": 0.5,
          "jpeg_quality": 20,
          "achieved_psnr": 22.19,
          "achieved_ssim": 0.496,
          "psnr_err": 0.39,
          "ssim_err": -0.0041,
          "within_tol": true
        }
      }
    }
  }
}''')
LEVELS = ['Level_1_Mild', 'Level_2_Moderate', 'Level_3_SevereCommon']
SEQUENCES = ['NBJ', 'NJB', 'BNJ', 'BJN', 'JNB', 'JBN']
FINAL_OPS = {'NBJ': 'JPEG', 'BNJ': 'JPEG', 'NJB': 'Blur', 'JNB': 'Blur', 'BJN': 'Noise', 'JBN': 'Noise'}

print('Loading Models on GPU...')
models = {
    'resnet50': timm.create_model('resnet50.a1_in1k', pretrained=True).eval().to(device),
    'convnext_tiny': timm.create_model('convnext_tiny.in12k_ft_in1k', pretrained=True).eval().to(device),
    'vit_base': timm.create_model('vit_base_patch16_224.augreg_in21k_ft_in1k', pretrained=True).eval().to(device),
    'swin_tiny': timm.create_model('swin_tiny_patch4_window7_224.ms_in1k', pretrained=True).eval().to(device),
}

model_norms = {
    'resnet50': T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    'convnext_tiny': T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    'vit_base': T.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
    'swin_tiny': T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
}

spatial_transform = T.Compose([T.Resize(256), T.CenterCrop(224), T.ToTensor()])
lpips_fn = lpips.LPIPS(net='alex').eval().to(device)

records = []
t0 = time.time()

for img_idx, item in enumerate(v2_dataset):
    with Image.open(item['filepath']) as pil_im:
        clean_img = spatial_transform(pil_im.convert('RGB'))
    true_lbl = item['class_idx']

    # Clean forward passes & records
    clean_margins = {}
    with torch.no_grad():
        for m_name, m_net in models.items():
            clean_norm = model_norms[m_name](clean_img).unsqueeze(0).to(device)
            logits = m_net(clean_norm).squeeze(0)
            probs = torch.softmax(logits, dim=0)
            z_t = logits[true_lbl].item()
            other_mask = torch.ones(logits.shape[0], dtype=torch.bool, device=logits.device)
            other_mask[true_lbl] = False
            z_o = logits[other_mask].max().item()
            clean_margin = z_t - z_o
            clean_margins[m_name] = clean_margin
            pred_c = int(torch.argmax(logits).item())
            records.append({
                'variant': item['variant'],
                'image_id': item['image_id'],
                'image_filename': item['filename'],
                'true_label': true_lbl,
                'wordnet_id': item['wordnet_id'],
                'class_name': item['class_name'],
                'level': 'Clean',
                'order': 'None',
                'final_op': 'None',
                'model': m_name,
                'psnr': 100.0,
                'ssim': 1.0,
                'lpips': 0.0,
                'fft_energy_low': 0.0,
                'fft_energy_med': 0.0,
                'fft_energy_high': 0.0,
                'jpeg_block_artifact': 1.0,
                'delta_edge_energy': 0.0,
                'std_local_ssim': 0.0,
                'logit_margin': round(clean_margin, 4),
                'delta_logit_margin': 0.0,
                'clean_logit_margin': round(clean_margin, 4),
                'top1_correct': bool(pred_c == true_lbl),
                'confidence': round(float(probs[pred_c].item()), 4),
            })

    # 18 Matched-Fidelity Conditions
    for lvl_idx, lvl in enumerate(LEVELS):
        for seq_idx, seq in enumerate(SEQUENCES):
            cfg = CALIBRATED_PARAMS['levels'][lvl]['sequences'][seq]
            seed = 42 + img_idx * 100 + lvl_idx * 10 + seq_idx
            corr_img = apply_compound_sequence(clean_img, order=seq, noise_std=cfg['noise_std'], blur_sigma=cfg['blur_sigma'], jpeg_quality=cfg['jpeg_quality'], noise_seed=seed)
            p_val = compute_psnr(clean_img, corr_img)
            s_val, _ = compute_ssim_and_local_map(clean_img, corr_img)
            with torch.no_grad():
                lp_clean = (clean_img.unsqueeze(0) * 2.0 - 1.0).to(device)
                lp_corr = (corr_img.unsqueeze(0) * 2.0 - 1.0).to(device)
                lpips_val = float(lpips_fn(lp_clean, lp_corr).item())
            fingerprints = compute_distortion_fingerprints(clean_img, corr_img)

            with torch.no_grad():
                for m_name, m_net in models.items():
                    corr_norm = model_norms[m_name](corr_img).unsqueeze(0).to(device)
                    logits = m_net(corr_norm).squeeze(0)
                    probs = torch.softmax(logits, dim=0)
                    z_t = logits[true_lbl].item()
                    other_mask = torch.ones(logits.shape[0], dtype=torch.bool, device=logits.device)
                    other_mask[true_lbl] = False
                    z_o = logits[other_mask].max().item()
                    margin = z_t - z_o
                    delta_margin = margin - clean_margins[m_name]
                    pred_cls = int(torch.argmax(logits).item())
                    records.append({
                        'variant': item['variant'],
                        'image_id': item['image_id'],
                        'image_filename': item['filename'],
                        'true_label': true_lbl,
                        'wordnet_id': item['wordnet_id'],
                        'class_name': item['class_name'],
                        'level': lvl,
                        'order': seq,
                        'final_op': FINAL_OPS[seq],
                        'model': m_name,
                        'psnr': round(p_val, 3),
                        'ssim': round(s_val, 4),
                        'lpips': round(lpips_val, 4),
                        **fingerprints,
                        'logit_margin': round(margin, 4),
                        'delta_logit_margin': round(delta_margin, 4),
                        'clean_logit_margin': round(clean_margins[m_name], 4),
                        'top1_correct': bool(pred_cls == true_lbl),
                        'confidence': round(float(probs[pred_cls].item()), 4),
                    })
    if (img_idx + 1) % 50 == 0 or (img_idx + 1) == len(v2_dataset):
        print(f'Progress: {img_idx + 1}/{len(v2_dataset)} images ({time.time() - t0:.1f}s)...')

df_all = pd.DataFrame(records)
df_all.to_csv('imagenet_v2_all_evaluations.csv.gz', index=False, compression='gzip')
print(f'Saved {len(df_all):,} ImageNet-V2 evaluations ({len(df_all[df_all["level"] == "Clean"])} clean + {len(df_all[df_all["level"] != "Clean"])} corrupted) in {time.time() - t0:.1f}s.')


In [ ]:
# 5. Statistical Analysis: Separate Variants & Pooled Analysis
def conventional_rcs_basis(x: np.ndarray, knots: np.ndarray) -> np.ndarray:
    K = len(knots)
    k_K, k_Km1 = knots[-1], knots[-2]
    basis = [x]
    denom_last = max(1e-8, abs(k_K - k_Km1))
    d_last = (np.maximum(0.0, x - k_Km1)**3 - np.maximum(0.0, x - k_K)**3) / denom_last
    for j in range(K - 2):
        k_j = knots[j]
        denom_j = max(1e-8, abs(k_K - k_j))
        d_j = (np.maximum(0.0, x - k_j)**3 - np.maximum(0.0, x - k_K)**3) / denom_j
        basis.append(d_j - d_last)
    return np.column_stack(basis)

def build_design_matrix(df_part, spec, scaler_params, spline_knots):
    N = len(df_part)
    orders = ['NBJ', 'NJB', 'BNJ', 'BJN', 'JNB', 'JBN']
    levels = ['Level_1_Mild', 'Level_2_Moderate', 'Level_3_SevereCommon']
    order_mat = np.zeros((N, len(orders) - 1))
    for i, o in enumerate(orders[1:]):
        order_mat[:, i] = (df_part['order'].to_numpy() == o).astype(float)
    level_mat = np.zeros((N, len(levels) - 1))
    for i, lvl in enumerate(levels[1:]):
        level_mat[:, i] = (df_part['level'].to_numpy() == lvl).astype(float)
    order_x_level = np.zeros((N, (len(orders) - 1) * (len(levels) - 1)))
    col_idx = 0
    for o_i in range(order_mat.shape[1]):
        for l_i in range(level_mat.shape[1]):
            order_x_level[:, col_idx] = order_mat[:, o_i] * level_mat[:, l_i]
            col_idx += 1
    p_std = (df_part['psnr'].to_numpy() - scaler_params['psnr']['mean']) / scaler_params['psnr']['std']
    s_std = (df_part['ssim'].to_numpy() - scaler_params['ssim']['mean']) / scaler_params['ssim']['std']
    l_std = (df_part['lpips'].to_numpy() - scaler_params['lpips']['mean']) / scaler_params['lpips']['std']
    intercept = np.ones((N, 1))
    base_parts = [intercept, order_mat, level_mat, order_x_level]
    p_sp = conventional_rcs_basis(p_std, spline_knots['psnr'])
    s_sp = conventional_rcs_basis(s_std, spline_knots['ssim'])
    l_sp = conventional_rcs_basis(l_std, spline_knots['lpips'])
    if 'extended' in spec:
        f_low = (df_part['fft_energy_low'].to_numpy() - scaler_params['fft_energy_low']['mean']) / scaler_params['fft_energy_low']['std']
        f_high = (df_part['fft_energy_high'].to_numpy() - scaler_params['fft_energy_high']['mean']) / scaler_params['fft_energy_high']['std']
        jpeg_b = (df_part['jpeg_block_artifact'].to_numpy() - scaler_params['jpeg_block_artifact']['mean']) / scaler_params['jpeg_block_artifact']['std']
        d_edge = (df_part['delta_edge_energy'].to_numpy() - scaler_params['delta_edge_energy']['mean']) / scaler_params['delta_edge_energy']['std']
        s_std_loc = (df_part['std_local_ssim'].to_numpy() - scaler_params['std_local_ssim']['mean']) / scaler_params['std_local_ssim']['std']
        fingerprints = np.column_stack([f_low, f_high, jpeg_b, d_edge, s_std_loc])
        return np.hstack(base_parts + [p_sp, s_sp, l_sp, fingerprints])
    return np.hstack(base_parts + [p_sp, s_sp, l_sp])

def analyze_dataset_subset(df_raw_sub, dataset_name, B=10000):
    df_sub = df_raw_sub[df_raw_sub['level'] != 'Clean'].copy()
    clusters = np.sort(df_sub['image_id'].unique())
    n_clusters = len(clusters)
    cluster_classes = df_sub.drop_duplicates('image_id').sort_values('image_id')['true_label'].to_numpy()
    rng_cv = np.random.RandomState(42)
    fold_assignments = np.zeros(n_clusters, dtype=int)
    for c in np.unique(cluster_classes):
        c_idx = np.where(cluster_classes == c)[0]
        perm = rng_cv.permutation(c_idx)
        c_splits = np.array_split(perm, 5)
        for f_i, s_idx in enumerate(c_splits): fold_assignments[s_idx] = f_i
    rng_b = np.random.RandomState(12345)
    b_choices = rng_b.randint(0, n_clusters, size=(B, n_clusters))
    W = np.zeros((B, n_clusters), dtype=float)
    for b in range(B): W[b, :] = np.bincount(b_choices[b], minlength=n_clusters)
    archs = ['convnext_tiny', 'vit_base', 'swin_tiny', 'resnet50']
    cont_cols = ['psnr', 'ssim', 'lpips', 'fft_energy_low', 'fft_energy_high', 'jpeg_block_artifact', 'delta_edge_energy', 'std_local_ssim']
    sub_res = []
    for arch in archs:
        df_arch = df_sub[df_sub['model'] == arch].copy().sort_values(['image_id', 'level', 'order']).reset_index(drop=True)
        y_true = df_arch['logit_margin'].to_numpy()
        N_eval = len(df_arch)
        p_c3, p_c5 = np.zeros(N_eval), np.zeros(N_eval)
        for fold_i in range(5):
            test_c = clusters[fold_assignments == fold_i]
            train_c = clusters[fold_assignments != fold_i]
            test_mask = df_arch['image_id'].isin(test_c).to_numpy()
            train_mask = df_arch['image_id'].isin(train_c).to_numpy()
            tr_df = df_arch[train_mask]
            te_df = df_arch[test_mask]
            scaler_params = {c: {'mean': float(np.mean(tr_df[c])), 'std': float(np.std(tr_df[c], ddof=1)) + 1e-8} for c in cont_cols}
            spline_knots = {c: np.percentile((tr_df[c] - scaler_params[c]['mean']) / scaler_params[c]['std'], [5.0, 35.0, 65.0, 95.0]) for c in ['psnr', 'ssim', 'lpips']}
            y_tr = tr_df['logit_margin'].to_numpy()
            X_tr_c3 = build_design_matrix(tr_df, 'conv', scaler_params, spline_knots)
            X_te_c3 = build_design_matrix(te_df, 'conv', scaler_params, spline_knots)
            X_tr_c5 = build_design_matrix(tr_df, 'conv_extended', scaler_params, spline_knots)
            X_te_c5 = build_design_matrix(te_df, 'conv_extended', scaler_params, spline_knots)
            b_c3 = np.linalg.pinv(X_tr_c3.T @ X_tr_c3) @ (X_tr_c3.T @ y_tr)
            b_c5 = np.linalg.pinv(X_tr_c5.T @ X_tr_c5) @ (X_tr_c5.T @ y_tr)
            p_c3[test_mask] = X_te_c3 @ b_c3
            p_c5[test_mask] = X_te_c5 @ b_c5
        ss_tot = np.sum((y_true - y_true.mean())**2)
        r2_3 = float(1.0 - np.sum((y_true - p_c3)**2) / ss_tot)
        r2_5 = float(1.0 - np.sum((y_true - p_c5)**2) / ss_tot)
        mae_3 = float(np.mean(np.abs(y_true - p_c3)))
        mae_5 = float(np.mean(np.abs(y_true - p_c5)))
        d_r2 = float(r2_5 - r2_3)
        d_mae = float(mae_3 - mae_5)
        # Bootstrap
        K_c = 18
        Y_c = y_true.reshape(n_clusters, K_c)
        P_c3 = p_c3.reshape(n_clusters, K_c)
        P_c5 = p_c5.reshape(n_clusters, K_c)
        c_s1, c_s2 = Y_c.sum(axis=1), (Y_c**2).sum(axis=1)
        b_sst = W @ c_s2 - ((W @ c_s1)**2) / float(N_eval)
        b_sse_3 = W @ (((Y_c - P_c3)**2).sum(axis=1))
        b_sse_5 = W @ (((Y_c - P_c5)**2).sum(axis=1))
        b_dr2 = (1.0 - b_sse_5 / b_sst) - (1.0 - b_sse_3 / b_sst)
        b_dmae = (W @ (np.abs(Y_c - P_c3).sum(axis=1))) / float(N_eval) - (W @ (np.abs(Y_c - P_c5).sum(axis=1))) / float(N_eval)
        ci_r2 = [float(np.percentile(b_dr2, 2.5)), float(np.percentile(b_dr2, 97.5))]
        ci_mae = [float(np.percentile(b_dmae, 2.5)), float(np.percentile(b_dmae, 97.5))]
        p_val = float((1.0 + np.sum((b_dr2 - d_r2) >= d_r2)) / (B + 1.0))
        sub_res.append({
            'variant': dataset_name,
            'architecture': arch,
            'n_images': n_clusters,
            'r2_m3': round(r2_3, 4),
            'r2_m5': round(r2_5, 4),
            'delta_r2': round(d_r2, 4),
            'ci95_delta_r2': [round(ci_r2[0], 4), round(ci_r2[1], 4)],
            'mae_m3': round(mae_3, 4),
            'mae_m5': round(mae_5, 4),
            'delta_mae': round(d_mae, 4),
            'ci95_delta_mae': [round(ci_mae[0], 4), round(ci_mae[1], 4)],
            'p_value': round(p_val, 5),
        })
        print(f'  {dataset_name:30s} | {arch:15s} | M3 R2: {r2_3:.4f}, M5 R2: {r2_5:.4f}, Delta R2: {d_r2:+.4f} (95% CI: [{ci_r2[0]:+.4f}, {ci_r2[1]:+.4f}]) | MAE Delta: {d_mae:+.4f} (p={p_val:.4f})')
    return sub_res

all_results = []
for v in ['MatchedFrequency', 'Threshold0.7', 'TopImages']:
    all_results.extend(analyze_dataset_subset(df_all[df_all['variant'] == v], v))
all_results.extend(analyze_dataset_subset(df_all, 'Pooled_External_All_Variants'))

df_res = pd.DataFrame(all_results)
df_res.to_csv('imagenet_v2_statistical_comparison.csv', index=False)
with open('imagenet_v2_replication_audit.json', 'w') as f: json.dump(all_results, f, indent=2)
manifest = {'title': 'ImageNet-V2 Replication Manifest', 'variants': ['MatchedFrequency', 'Threshold0.7', 'TopImages'], 'total_records': len(df_all), 'unique_images': df_all['image_id'].nunique(), 'gpu_device': str(device), 'status': 'COMPLETE'}
with open('imagenet_v2_execution_manifest.json', 'w') as f: json.dump(manifest, f, indent=2)
print('Saved all ImageNet-V2 replication results!')
